# Preparation

In [1]:
import pandas as pd
import sys
sys.path.append('../../')
from MEGA_utilities import data_col_standardize, data_remove_duplicate


# load data
Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H1N1['Oseltamivir']= '<NOSV>'
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
Crick_H3N2['Oseltamivir']= '<OSV>'
origin_df = pd.concat([Crick_H1N1, Crick_H3N2]).reset_index(drop=True)

In [2]:
## select required columns
AA_data_filt1 = origin_df[['serumName','virusName','serumHA', 'serumNA', 'virusHA', 'virusNA', 
                           'serumPassCat','virusPassCat', 'Oseltamivir', 'serumType','HI_Dist']].copy()
## remove duplicated row and mean HI_Dist
AA_data_filt2 = AA_data_filt1.groupby(['serumHA', 'serumNA', 'virusHA', 'virusNA', 'serumPassCat', 'virusPassCat', 'Oseltamivir']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA',
                        'serumPassCat', 'virusPassCat', 'Oseltamivir', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
AA_data_filt3 = AA_data_filt2[(AA_data_filt2['serumPassCat'] != 'BOTH') &
                              (AA_data_filt2['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
AA_data_filt4 = AA_data_filt3.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                       'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

AA_data_final = AA_data_filt4.copy()

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

class AADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA'] + '<eos>' + DataFrame['serumNA'] + '<eos>' + DataFrame['virusHA'] + '<eos>' + DataFrame['virusNA'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>' + DataFrame['Oseltamivir']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(AA_data_filt4, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = AADataset(train_df)
valid_dataset = AADataset(valid_df)
test_dataset = AADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [5]:
# train_df.to_csv('../../../data/processed/1.1/AA_train_df.csv', index=True)
# valid_df.to_csv('../../../data/processed/1.1/AA_valid_df.csv', index=True)
# test_df.to_csv('../../../data/processed/1.1/AA_test_df.csv', index=True)

In [6]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=30
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=5
device = torch.device("cuda:0")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 1.135691e+06


In [7]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

save_path = '../../../trained_model/1.4_Meta_info/'
progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir=save_path)

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open(save_path + 'log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 5761/921760 [14:19<36:38:20,  6.94it/s]

train loss : 2.924137557417832
MAE:  1.266079874150274
MSE:  2.766639684644268
pearson correlation:  PearsonRResult(statistic=np.float64(0.5230938669171441), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.5598580215259686), pvalue=np.float64(0.0))
Validation MSE decrease (inf --> 2.766640).  Saving model ...


  1%|▏         | 11522/921760 [30:02<35:59:05,  7.03it/s] 

train loss : 2.74117744452262


  1%|▏         | 11523/921760 [31:21<6014:27:28, 23.79s/it]

...

train loss : 0.6405791053223518


 94%|█████████▍| 869912/921760 [39:35:40<343:02:08, 23.82s/it]

MAE:  0.6981302126310895
MSE:  0.831136860002453
pearson correlation:  PearsonRResult(statistic=np.float64(0.8843701169331117), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.8541443469266798), pvalue=np.float64(0.0))
EarlyStopping counter: 9 out of 10


 95%|█████████▌| 875672/921760 [39:49:58<1:49:10,  7.04it/s]  

train loss : 0.6380978275392437
MAE:  0.6981948278570171
MSE:  0.8298961285001951
pearson correlation:  PearsonRResult(statistic=np.float64(0.8841723740541609), pvalue=np.float64(0.0))
spearman correlation:  SignificanceResult(statistic=np.float64(0.8539675068091631), pvalue=np.float64(0.0))
EarlyStopping counter: 10 out of 10
Early stopping
